# 3. Train CatBoost & Index for Vector Search

All learned/indexed classifiers in one notebook. Each section appends
rows to `cat_predictions` with a distinct `source` label so the app can
compare methods.

Sections:
1. **CatBoost** (3-level only) — supervised hierarchical classifier on
   the bootstrap labels. Source: `catboost`.
2. **UNSPSC pgvector** — hybrid (cosine + tsvector) search across 150k
   commodity codes, embeddings stored in Lakebase. Source: `pgvector`.

Skip a section by leaving its cells un-run; nothing is required for the
next step to work.

In [ ]:
%pip install pyyaml openai openpyxl pydantic catboost scikit-learn pgvector psycopg2-binary
%restart_python

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_spark
from src.config import load_config
import pyspark.sql.functions as F
import json, uuid

spark = get_spark()
config = load_config()

In [ ]:
dbutils.widgets.removeAll()
dbutils.widgets.text('catalog', config.catalog)
dbutils.widgets.text('schema', config.schema_name)
dbutils.widgets.text('invoices_table', config.invoices)
dbutils.widgets.text('lakebase_instance', config.lakebase_instance)
dbutils.widgets.text('embed_endpoint', 'databricks-bge-large-en')
dbutils.widgets.text('embed_dim', '1024')
dbutils.widgets.text('top_k', '20')
dbutils.widgets.text('unspsc_sample', '200')
dbutils.widgets.text('run_catboost', 'true')
dbutils.widgets.text('run_pgvector', 'true')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
invoices_table = dbutils.widgets.get('invoices_table')
lakebase_instance = dbutils.widgets.get('lakebase_instance')
embed_endpoint = dbutils.widgets.get('embed_endpoint')
embed_dim = int(dbutils.widgets.get('embed_dim'))
top_k = int(dbutils.widgets.get('top_k'))
unspsc_sample = int(dbutils.widgets.get('unspsc_sample'))
run_catboost = dbutils.widgets.get('run_catboost').lower() == 'true'
run_pgvector = dbutils.widgets.get('run_pgvector').lower() == 'true'
spark.sql(f'USE {catalog}.{schema}')

## 1. CatBoost (3-level only)

Trains a hierarchical L1 → L2 classifier on the bootstrap labels and
writes predictions back to `cat_predictions` with `source='catboost'`.

In [ ]:
if run_catboost:
    import pandas as pd
    from catboost import CatBoostClassifier
    from sklearn.model_selection import train_test_split

    df = spark.sql(f"""
      SELECT i.order_id,
             CONCAT_WS(' | ', i.supplier, i.description, i.cost_centre) AS text,
             p.level_path[0] AS l1, p.level_path[1] AS l2
      FROM {catalog}.{schema}.{invoices_table} i
      JOIN {catalog}.{schema}.cat_predictions p
        ON p.order_id = i.order_id AND p.schema_name = 'three_level' AND p.source = 'ai_classify'
      WHERE p.level_path[0] IS NOT NULL AND p.level_path[1] IS NOT NULL
    """).toPandas()
    print(f'CatBoost training on {len(df):,} bootstrapped rows')

    train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df['l1'])

    m_l1 = CatBoostClassifier(iterations=200, verbose=0, text_features=['text'])
    m_l1.fit(train[['text']], train['l1'])
    m_l2 = CatBoostClassifier(iterations=200, verbose=0, text_features=['text'])
    m_l2.fit(train[['text']], train['l2'])

    test_pred = test.copy()
    test_pred['pred_l1'] = m_l1.predict(test[['text']]).flatten()
    test_pred['pred_l2'] = m_l2.predict(test[['text']]).flatten()
    test_pred['confidence'] = m_l2.predict_proba(test[['text']]).max(axis=1)

    # Join L3 default from taxonomy (optional, leaves blank if not unique)
    out = pd.DataFrame({
        'order_id': test_pred['order_id'].astype(str),
        'schema_name': 'three_level',
        'code': test_pred['pred_l2'],
        'label': test_pred['pred_l2'],
        'level_path': [[a, b] for a, b in zip(test_pred['pred_l1'], test_pred['pred_l2'])],
        'confidence': test_pred['confidence'].astype(float),
        'rationale': 'CatBoost classifier predict_proba',
        'source': 'catboost',
        'candidates': [[] for _ in range(len(test_pred))],
    })
    sdf = spark.createDataFrame(out).withColumn('classified_at', F.current_timestamp())
    sdf.createOrReplaceTempView('_catboost_predictions')
    spark.sql(f"""
    MERGE INTO {catalog}.{schema}.cat_predictions t
    USING _catboost_predictions s
    ON t.order_id = s.order_id AND t.schema_name = s.schema_name AND t.source = s.source
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    """)
    print(f'Merged {len(out):,} catboost predictions')
else:
    print('Skipping CatBoost (run_catboost=false)')

## 2. UNSPSC tsvector search (Postgres full-text, no embeddings)

Loads ~150k UNSPSC commodities into a Lakebase native table with a GIN
`tsvector` index, then ranks invoices against it with `ts_rank_cd`.
This avoids the cost of embedding 150k codes; full-text search is good
enough when invoice descriptions share vocabulary with UNSPSC titles.

In [ ]:
if run_pgvector:
    # 1. Build the per-commodity search text in Delta.
    spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.{schema}.unspsc_search AS
    SELECT code, label, level_path,
      CONCAT_WS(' | ', label, class_title, family_title, segment_title,
                       commodity_definition, class_definition, synonym) AS search_text
    FROM {catalog}.{schema}.unspsc_taxonomy
    """)
    print('unspsc_search built:', spark.table(f'{catalog}.{schema}.unspsc_search').count())

In [ ]:
if run_pgvector:
    from databricks.sdk import WorkspaceClient
    import psycopg2
    from psycopg2.extras import execute_values

    w = WorkspaceClient()

    def pg_connect():
        inst = w.database.get_database_instance(name=lakebase_instance)
        cred = w.database.generate_database_credential(
            request_id=str(uuid.uuid4()), instance_names=[lakebase_instance]
        )
        me = w.current_user.me()
        user = me.emails[0].value if me.emails else me.user_name
        return psycopg2.connect(host=inst.read_write_dns, dbname=config.lakebase_dbname,
                                user=user, password=cred.token, sslmode='require')

    SP_ROLE = 'aab57292-8e82-48bf-8e43-e8f43fe2743f'  # spend-app SP

    with pg_connect() as conn, conn.cursor() as cur:
        cur.execute(f'CREATE SCHEMA IF NOT EXISTS "{schema}"')
        cur.execute(f"""
        CREATE TABLE IF NOT EXISTS "{schema}".unspsc_text (
          code TEXT PRIMARY KEY, label TEXT, level_path TEXT[],
          search_text TEXT, tsv tsvector
        )""")
        cur.execute(f'CREATE INDEX IF NOT EXISTS unspsc_text_tsv_idx ON "{schema}".unspsc_text USING GIN (tsv)')
        conn.commit()

    BATCH = 1000
    iterator = (spark.table(f'{catalog}.{schema}.unspsc_search')
                     .select('code','label','level_path','search_text').toLocalIterator())
    with pg_connect() as conn, conn.cursor() as cur:
        batch, n = [], 0
        for row in iterator:
            batch.append((row['code'], row['label'], list(row['level_path'] or []), row['search_text']))
            if len(batch) >= BATCH:
                execute_values(cur, f"""
                  INSERT INTO "{schema}".unspsc_text (code, label, level_path, search_text) VALUES %s
                  ON CONFLICT (code) DO UPDATE SET label=EXCLUDED.label,
                    level_path=EXCLUDED.level_path, search_text=EXCLUDED.search_text
                """, batch, template='(%s,%s,%s,%s)')
                conn.commit(); n += len(batch); batch.clear()
        if batch:
            execute_values(cur, f"""
              INSERT INTO "{schema}".unspsc_text (code, label, level_path, search_text) VALUES %s
              ON CONFLICT (code) DO UPDATE SET label=EXCLUDED.label,
                level_path=EXCLUDED.level_path, search_text=EXCLUDED.search_text
            """, batch, template='(%s,%s,%s,%s)')
            conn.commit(); n += len(batch)
        cur.execute(f"""UPDATE "{schema}".unspsc_text SET tsv = to_tsvector('english', search_text) WHERE tsv IS NULL""")
        conn.commit()
        # Grant SP access so the app can read it (not strictly needed for this run; safety).
        try:
            cur.execute(f'GRANT SELECT ON "{schema}".unspsc_text TO "{SP_ROLE}"')
            conn.commit()
        except Exception:
            pass
        cur.execute(f'SELECT count(*) FROM "{schema}".unspsc_text')
        print('Lakebase unspsc_text rows:', cur.fetchone()[0])

In [ ]:
if run_pgvector:
    import re
    from datetime import datetime
    from pyspark.sql.types import (StructType, StructField, StringType,
                                   ArrayType, DoubleType, TimestampType)

    invoices = spark.sql(f"""
      SELECT order_id, supplier, description, cost_centre,
             CONCAT_WS(' | ', supplier, description, cost_centre) AS qtext
      FROM {catalog}.{schema}.{invoices_table}
      ORDER BY order_id
    """).toPandas()
    print(f'Classifying {len(invoices)} invoices via tsvector')

    # Drop generic / boilerplate words that match too many UNSPSC entries.
    STOPWORDS = {
        'supplier','description','cost','centre','center','for','use','and','the',
        'of','with','on','services','service','products','product','group','company',
        'industries','solutions','systems','inc','llc','corp','co','ltd','plc',
    }

    def or_tsquery(text: str) -> str:
        toks = []
        seen = set()
        for t in re.split(r'[^A-Za-z]+', text or ''):
            tl = t.lower()
            if len(tl) >= 4 and tl not in STOPWORDS and tl not in seen:
                toks.append(tl)
                seen.add(tl)
        return ' | '.join(toks)

    TOPK_SQL = f"""
    SELECT code, label, level_path, ts_rank(tsv, q, 1) AS score
    FROM "{schema}".unspsc_text, to_tsquery('english', %s) q
    WHERE tsv @@ q
    ORDER BY ts_rank(tsv, q, 1) DESC
    LIMIT {top_k};
    """

    rows = []
    misses = 0
    now = datetime.utcnow()
    with pg_connect() as conn, conn.cursor() as cur:
        for _, r in invoices.iterrows():
            tsq = or_tsquery(r['qtext'])
            if not tsq:
                misses += 1; continue
            try:
                cur.execute(TOPK_SQL, (tsq,))
                res = cur.fetchall()
            except Exception:
                conn.rollback(); misses += 1; continue
            if not res:
                misses += 1; continue
            cands = [
                {'code': c[0], 'label': c[1], 'score': float(c[3])}
                for c in res
            ]
            top_code, top_label, top_lp, top_score = res[0]
            rows.append({
                'order_id': str(r['order_id']),
                'schema_name': 'unspsc',
                'code': top_code,
                'label': top_label,
                'level_path': list(top_lp or []),
                'confidence': float(top_score),
                'rationale': f'tsvector OR-token ts_rank top-1 of {len(cands)} hits',
                'source': 'tsvector',
                'candidates': cands,
                'classified_at': now,
            })

    print(f'Predictions for {len(rows)}/{len(invoices)} invoices (misses: {misses})')

    if rows:
        pred_schema = StructType([
            StructField('order_id', StringType()), StructField('schema_name', StringType()),
            StructField('code', StringType()), StructField('label', StringType()),
            StructField('level_path', ArrayType(StringType())),
            StructField('confidence', DoubleType()), StructField('rationale', StringType()),
            StructField('source', StringType()),
            StructField('candidates', ArrayType(StructType([
                StructField('code', StringType()), StructField('label', StringType()),
                StructField('score', DoubleType()),
            ]))),
            StructField('classified_at', TimestampType()),
        ])
        sdf = spark.createDataFrame(rows, schema=pred_schema)
        sdf.createOrReplaceTempView('_unspsc_predictions')
        spark.sql(f"""
        MERGE INTO {catalog}.{schema}.cat_predictions t
        USING _unspsc_predictions s
        ON t.order_id = s.order_id AND t.schema_name = s.schema_name AND t.source = s.source
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """)
        spark.sql(f"SELECT order_id, code, label, confidence FROM {catalog}.{schema}.cat_predictions WHERE schema_name='unspsc' ORDER BY confidence DESC LIMIT 10").display()